# 05 — Memory Store: del Goldfish al CFO Interno

**Autor:** Gabriel Untiveros | **Fecha:** 2026-05-26  
**Requiere:** NB 04 ejecutado (loop agentico base funcional)

---

## El problema del NB 04

El agente del NB 04 funciona. Pero tiene memoria de pez dorado:

```
Sesión lunes:  "CUENTA-003 muestra caída semanal de -USD 3,235"
Sesión martes: "¿Cambió algo respecto al lunes?"
Agente martes: "No sé nada del lunes."
```

Eso no es un CFO interno. Es un analista que olvida todo entre reuniones.

## Lo que construimos aquí

Un **Memory Store** con SQLite que persiste entre sesiones:

| Tabla | Para qué |
|---|---|
| `sessions` | Historial de consultas y resúmenes |
| `alertas` | Estado de cuentas detectado en cada sesión |
| `client_facts` | Preferencias y umbrales configurados por el cliente |
| `pending_actions` | Acciones recomendadas pendientes de completar |

El agente con memoria puede decir:

> "La semana pasada CUENTA-003 mostró una caída de -USD 3,235. Hoy veo +USD 187/día. 
> El problema se resolvió, pero el calendario de pagos USD que recomendé revisar sigue pendiente."

Ese es el diferenciador comercial.

## Arquitectura del NB 05

```
INICIO DE SESIÓN:
  memory.build_context_block()    ← carga historial
         ↓
  system_prompt + context_block   ← agente arranca con contexto

DURANTE LA SESIÓN:
  run_agent_with_memory()          ← loop igual al NB 04 + fix WinError 2
  tool_log guarda outputs          ← nuevo vs NB 04

FIN DE SESIÓN:
  extract_account_state(tool_log)  ← parsea JSONs de bash_execute
  memory.save_session()            ← guarda resumen
  memory.save_alertas()            ← guarda estados de cuentas
```

In [1]:
import json, subprocess, sys, os, sqlite3, uuid
from pathlib import Path
from datetime import datetime

# ── API key desde .env ────────────────────────────────────────────────────
env_file = Path('..') / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

import anthropic

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
assert api_key.startswith('sk-ant-'), (
    'ANTHROPIC_API_KEY no encontrada.\n'
    'Crea Skill_financiero/.env con: ANTHROPIC_API_KEY=sk-ant-...'
)

# ── Paths ─────────────────────────────────────────────────────────────────
BASE        = Path('..').resolve()
SKILLS_BASE = BASE / '.claude' / 'skills'
MEMORY_DIR  = BASE / 'data' / 'memory'
MEMORY_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_DB   = MEMORY_DIR / 'agente_tesoreria.db'

# ── Verificar prerequisitos ───────────────────────────────────────────────
required = [
    SKILLS_BASE / 'forecast-cashflow' / 'batch_dias_de_caja.py',
    SKILLS_BASE / 'alerta-tesoreria'  / 'SKILL.md',
    SKILLS_BASE / 'reporte-semanal'   / 'generar_reporte.py',
]
for r in required:
    assert r.exists(), f'Falta {r}. Ejecuta primero NB 03.'

# ── Cliente Anthropic ─────────────────────────────────────────────────────
MODEL  = 'claude-sonnet-4-6'
client = anthropic.Anthropic(api_key=api_key)

print(f'Base del proyecto: {BASE}')
print(f'Memory DB:         {MEMORY_DB}')
print(f'Modelo:            {MODEL}')
print(f'Skills:            {[p.parent.name for p in SKILLS_BASE.glob("*/SKILL.md")]}')

Base del proyecto: D:\Proyecto_Gabriel\Skill_financiero
Memory DB:         D:\Proyecto_Gabriel\Skill_financiero\data\memory\agente_tesoreria.db
Modelo:            claude-sonnet-4-6
Skills:            ['alerta-tesoreria', 'forecast-cashflow', 'reporte-semanal']


---
## MemoryStore — el cerebro persistente del agente

**Backend:** SQLite — cero dependencias externas, archivo único, portable.  
**Diseño:** Cada sesión es un registro. Los estados de cuentas se guardan por separado para poder comparar evolución.

El método clave es `build_context_block()` — construye el texto que se inyecta en el system prompt al inicio de cada sesión.

In [2]:
class MemoryStore:
    """
    Almacén de memoria persistente para el agente de tesorería.

    Tablas:
      sessions        — historial de consultas y resúmenes
      alertas         — estados de cuentas detectados por sesión
      client_facts    — preferencias/umbrales del cliente (key-value)
      pending_actions — acciones recomendadas pendientes de completar
    """

    def __init__(self, db_path: Path):
        self.db_path = Path(db_path)
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.executescript("""
                CREATE TABLE IF NOT EXISTS sessions (
                    id          TEXT PRIMARY KEY,
                    fecha       TEXT NOT NULL,
                    query       TEXT NOT NULL,
                    summary     TEXT,
                    turns       INTEGER,
                    tokens_in   INTEGER,
                    tokens_out  INTEGER
                );

                CREATE TABLE IF NOT EXISTS alertas (
                    id           INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id   TEXT    NOT NULL,
                    cuenta_id    TEXT    NOT NULL,
                    nivel        TEXT    NOT NULL,
                    saldo_actual REAL,
                    dias_de_caja REAL,
                    flujo_neto   REAL,
                    fecha        TEXT    NOT NULL,
                    FOREIGN KEY (session_id) REFERENCES sessions(id)
                );

                CREATE TABLE IF NOT EXISTS client_facts (
                    key        TEXT PRIMARY KEY,
                    value      TEXT NOT NULL,
                    updated_at TEXT NOT NULL
                );

                CREATE TABLE IF NOT EXISTS pending_actions (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    accion     TEXT NOT NULL,
                    cuenta_id  TEXT,
                    completed  INTEGER DEFAULT 0,
                    created_at TEXT    NOT NULL,
                    FOREIGN KEY (session_id) REFERENCES sessions(id)
                );
            """)

    # ── WRITE ────────────────────────────────────────────────────────────────

    def save_session(self, session_id: str, query: str, result: dict) -> None:
        """Guarda el resumen de una sesión completa."""
        summary = (result.get('final_text') or '')[:300].replace('\n', ' ')
        fecha   = datetime.utcnow().isoformat()
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                'INSERT OR REPLACE INTO sessions VALUES (?,?,?,?,?,?,?)',
                (session_id, fecha, query, summary,
                 result.get('turns'), result.get('tokens_in'), result.get('tokens_out'))
            )

    def save_alertas(self, session_id: str, states: list) -> None:
        """Guarda los estados de cuentas extraídos del tool_log."""
        fecha = datetime.utcnow().isoformat()
        with sqlite3.connect(self.db_path) as conn:
            for s in states:
                conn.execute(
                    'INSERT INTO alertas '
                    '(session_id, cuenta_id, nivel, saldo_actual, dias_de_caja, flujo_neto, fecha) '
                    'VALUES (?,?,?,?,?,?,?)',
                    (session_id, s.get('cuenta_id', ''), s.get('nivel', 'OK'),
                     s.get('saldo_actual'), s.get('dias_de_caja'), s.get('flujo_neto_dia'), fecha)
                )

    def save_fact(self, key: str, value: str) -> None:
        """Guarda o actualiza un hecho del cliente. Ej: ('CUENTA-002.umbral', '10 dias')."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                'INSERT OR REPLACE INTO client_facts VALUES (?,?,?)',
                (key, value, datetime.utcnow().isoformat())
            )

    def save_pending_action(
        self, session_id: str, accion: str, cuenta_id: str = None
    ) -> None:
        """Registra una acción recomendada (la marca el tesorero, no el agente)."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                'INSERT INTO pending_actions (session_id, accion, cuenta_id, created_at) '
                'VALUES (?,?,?,?)',
                (session_id, accion, cuenta_id, datetime.utcnow().isoformat())
            )

    def mark_completed(self, action_id: int) -> None:
        """Marca una acción como completada."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('UPDATE pending_actions SET completed=1 WHERE id=?', (action_id,))

    # ── READ ─────────────────────────────────────────────────────────────────

    def get_recent_sessions(self, n: int = 3) -> list:
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                'SELECT * FROM sessions ORDER BY fecha DESC LIMIT ?', (n,)
            ).fetchall()
            return [dict(r) for r in rows]

    def get_last_state_per_account(self) -> list:
        """Último estado conocido de cada cuenta (JOIN para evitar duplicados)."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute("""
                SELECT a.* FROM alertas a
                INNER JOIN (
                    SELECT cuenta_id, MAX(fecha) AS max_fecha
                    FROM alertas GROUP BY cuenta_id
                ) latest ON a.cuenta_id = latest.cuenta_id
                         AND a.fecha    = latest.max_fecha
                ORDER BY a.nivel DESC, a.cuenta_id
            """).fetchall()
            return [dict(r) for r in rows]

    def get_pending_actions(self) -> list:
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                'SELECT * FROM pending_actions WHERE completed=0 '
                'ORDER BY created_at DESC LIMIT 5'
            ).fetchall()
            return [dict(r) for r in rows]

    def get_client_facts(self) -> dict:
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute('SELECT key, value FROM client_facts').fetchall()
            return {r[0]: r[1] for r in rows}

    def count_sessions(self) -> int:
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute('SELECT COUNT(*) FROM sessions').fetchone()[0]

    # ── CONTEXT INJECTION ────────────────────────────────────────────────────

    def build_context_block(self) -> str:
        """
        Construye el bloque de contexto histórico para inyectar en el system prompt.
        Retorna '' si es la primera sesión (sin historial).

        Incluye:
          - Últimas 3 sesiones (fecha + query + resumen corto)
          - Último estado de alerta por cuenta
          - Acciones pendientes (no completadas)
          - Configuración del cliente (client_facts)
        """
        sessions = self.get_recent_sessions(3)
        if not sessions:
            return ''   # Primera sesión — sin contexto previo

        states  = self.get_last_state_per_account()
        pending = self.get_pending_actions()
        facts   = self.get_client_facts()

        lines = [
            '',
            '## HISTORIAL DEL CLIENTE (referencia interna — no citar textualmente al usuario)',
            '',
            '### Sesiones recientes:',
        ]
        for s in sessions:
            fecha_str = s['fecha'][:10]
            summary   = (s['summary'] or '')[:150].replace('\n', ' ')
            lines.append(f'- [{fecha_str}] "{s["query"]}" → {summary}')

        if states:
            lines.append('')
            lines.append('### Último estado conocido por cuenta:')
            nivel_emoji = {'CRITICO': '🔴', 'ALTO': '🟠', 'MEDIO': '🟡', 'OK': '🟢'}
            for a in states:
                emoji = nivel_emoji.get(a['nivel'], '⚪')
                flujo = a['flujo_neto'] or 0.0
                saldo = a['saldo_actual'] or 0.0
                lines.append(
                    f"  {a['cuenta_id']}: {emoji} {a['nivel']} | "
                    f"Saldo: {saldo:,.0f} | "
                    f"Flujo/día: {flujo:+.0f} | "
                    f"Registrado: {a['fecha'][:10]}"
                )

        if pending:
            lines.append('')
            lines.append('### Acciones pendientes (recomendadas, aún no completadas):')
            for p in pending:
                cuenta_str = f" [{p['cuenta_id']}]" if p['cuenta_id'] else ''
                lines.append(
                    f"  - {p['accion']}{cuenta_str} (desde: {p['created_at'][:10]})"
                )

        if facts:
            lines.append('')
            lines.append('### Configuración del cliente:')
            for k, v in facts.items():
                lines.append(f'  - {k}: {v}')

        lines.append('')
        return '\n'.join(lines)

    # ── UTILITY ──────────────────────────────────────────────────────────────

    def __repr__(self):
        n = self.count_sessions()
        return f'<MemoryStore db={self.db_path.name} sessions={n}>'


print('Clase MemoryStore definida.')

Clase MemoryStore definida.


In [3]:
# Inicializar (crea el .db si no existe, no hace nada si ya existe)
memory = MemoryStore(MEMORY_DB)
print(memory)

ctx = memory.build_context_block()
if ctx:
    print(f'Contexto existente ({memory.count_sessions()} sesión/es previas):')
    print(ctx)
else:
    print('Contexto vacío — primera sesión.')

<MemoryStore db=agente_tesoreria.db sessions=0>
Contexto vacío — primera sesión.


---
## Dispatch v2 — eliminamos el WinError 2

En NB 04, el modelo pasaba `command` como JSON string en el primer turn:  
`"[\"python\", \"script.py\"]"` en vez de `["python", "script.py"]`

Costaba 1 turn extra por query. Fix: detectar el tipo antes de llamar `subprocess.run()`.

In [4]:
TOOL_DEFS = [
    {
        'name': 'bash_execute',
        'description': (
            'Ejecuta un script Python de los skills de tesoreria. '
            'Usa esto para: forecast de caja (batch_dias_de_caja.py), '
            'forecast individual (rolling_mean_cashflow.py CUENTA-ID 14), '
            'reporte semanal (generar_reporte.py). '
            'El comando DEBE ser una lista Python de strings, no un string JSON.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'command': {
                    'type': 'array',
                    'items': {'type': 'string'},
                    'description': 'Lista de strings. Ej: ["python", "ruta/script.py"]'
                }
            },
            'required': ['command']
        }
    },
    {
        'name': 'read_skill',
        'description': (
            'Lee el contenido de un SKILL.md para cargar la politica bajo demanda. '
            'Skills disponibles: forecast-cashflow/SKILL.md, '
            'alerta-tesoreria/SKILL.md, reporte-semanal/SKILL.md.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'skill_name': {
                    'type': 'string',
                    'description': 'Nombre del skill. Ej: alerta-tesoreria/SKILL.md'
                }
            },
            'required': ['skill_name']
        }
    }
]


def dispatch_v2(tool_name: str, tool_input: dict) -> str:
    """
    Ejecuta una tool call. Mejora sobre dispatch() del NB 04:
    - Maneja el caso donde 'command' llega como JSON string (bug del modelo)
    """
    if tool_name == 'bash_execute':
        command = tool_input['command']

        # Fix WinError 2 ─────────────────────────────────────────────────────
        # El modelo a veces pasa command como JSON string en el primer turn.
        # Ej: "[\"python\", \"D:/...script.py\"]" en vez de ["python", "D:/..."]
        if isinstance(command, str):
            try:
                command = json.loads(command)
            except json.JSONDecodeError:
                command = command.split()   # fallback: split por espacios
        # ────────────────────────────────────────────────────────────────────

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            cwd=str(BASE),
            timeout=30
        )
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            return f'ERROR (exit {result.returncode}): {result.stderr.strip()}'

    elif tool_name == 'read_skill':
        skill_path = SKILLS_BASE / tool_input['skill_name']
        if skill_path.exists():
            return skill_path.read_text(encoding='utf-8')
        else:
            return f'ERROR: Skill no encontrado: {skill_path}'

    return f'ERROR: Tool desconocida: {tool_name}'


print('dispatch_v2() listo.')
print('Tools:', [t['name'] for t in TOOL_DEFS])

dispatch_v2() listo.
Tools: ['bash_execute', 'read_skill']


---
## Extracción de estados — parsear los outputs del tool_log

En NB 04, `tool_log` solo guardaba `{turn, tool, input}` — el output se perdía.  
En NB 05, guardamos también el `output` para poder parsear los JSONs post-sesión.

La función `extract_account_state()` extrae los estados de cuentas de los outputs de `bash_execute` sin hacer una llamada extra a la API.

In [5]:
def extract_account_state(tool_log: list) -> list:
    """
    Extrae el estado de cuentas de los outputs de bash_execute.

    Parsea el JSON de cada output. Infiere nivel de alerta a partir de
    dias_de_caja + requiere_atencion. Deduplica por cuenta_id.

    Retorna: lista de dicts con {cuenta_id, nivel, saldo_actual, dias_de_caja, flujo_neto_dia}
    """
    states = []
    seen   = set()

    for entry in tool_log:
        if entry.get('tool') != 'bash_execute':
            continue
        output = entry.get('output', '')
        if not output or output.startswith('ERROR'):
            continue

        try:
            data = json.loads(output)
        except (json.JSONDecodeError, TypeError):
            # El output puede ser markdown (reporte). Ignorar.
            continue

        items = data if isinstance(data, list) else [data]

        for item in items:
            if not isinstance(item, dict) or 'cuenta_id' not in item:
                continue
            cid = item['cuenta_id']
            if cid in seen:
                continue
            seen.add(cid)

            # Inferir nivel de alerta ─────────────────────────────────────────
            # La política completa está en alerta-tesoreria/SKILL.md.
            # Aquí usamos la regla simplificada para el registro en memoria.
            dias         = item.get('dias_de_caja', 999)
            requiere_at  = item.get('requiere_atencion', False)

            if not requiere_at or dias == 999:
                nivel = 'OK'
            elif dias < 5:
                nivel = 'CRITICO'
            elif dias < 10:
                nivel = 'ALTO'
            elif dias < 20:
                nivel = 'MEDIO'
            else:
                nivel = 'OK'
            # ──────────────────────────────────────────────────────────────────

            states.append({
                'cuenta_id'     : cid,
                'nivel'         : nivel,
                'saldo_actual'  : item.get('saldo_actual', 0.0),
                'dias_de_caja'  : dias,
                'flujo_neto_dia': item.get('flujo_neto_dia', 0.0),
            })

    return states


print('extract_account_state() listo.')

extract_account_state() listo.


---
## System prompt + run_agent_with_memory()

El loop es idéntico al NB 04 con 3 diferencias:
1. El system prompt incluye el bloque de contexto histórico
2. El tool_log guarda el output de cada tool (no solo el input)
3. Al finalizar, guarda sesión + estados de cuentas en memoria

In [6]:
SYSTEM_PROMPT_BASE = f'''
Eres un agente de tesoreria financiero con memoria de sesiones previas.
Cuando el historial del cliente esté disponible (sección HISTORIAL DEL CLIENTE),
úsalo para dar contexto comparativo: qué cambió, qué se resolvió, qué sigue pendiente.

HERRAMIENTAS:
- bash_execute: ejecuta scripts Python de los skills (command DEBE ser lista Python)
- read_skill: lee un SKILL.md para cargar la politica cuando la necesitas

SCRIPTS DISPONIBLES (usar con bash_execute):
  Todas las cuentas:  ["{sys.executable}", "{(SKILLS_BASE / 'forecast-cashflow' / 'batch_dias_de_caja.py').as_posix()}"]
  Cuenta individual:  ["{sys.executable}", "{(SKILLS_BASE / 'forecast-cashflow' / 'rolling_mean_cashflow.py').as_posix()}", "CUENTA-ID", "14"]
  Reporte semanal:    ["{sys.executable}", "{(SKILLS_BASE / 'reporte-semanal' / 'generar_reporte.py').as_posix()}"]

SKILLS (cargar con read_skill cuando la tarea lo requiera):
  forecast-cashflow/SKILL.md  -> cuando pidan forecast o proyeccion
  alerta-tesoreria/SKILL.md   -> cuando pidan alertas o revision de liquidez
  reporte-semanal/SKILL.md    -> cuando pidan el reporte semanal

REGLA PRINCIPAL: Usa UN script para procesar datos de todas las cuentas.
No hagas llamadas individuales por cuenta — ese es el anti-patron.
'''.strip()

print(f'System prompt base: {len(SYSTEM_PROMPT_BASE.splitlines())} líneas')

System prompt base: 20 líneas


In [7]:
def run_agent_with_memory(
    prompt: str,
    memory: MemoryStore,
    max_turns: int = 10,
    verbose: bool = True,
) -> dict:
    """
    Loop agentico con Memory Store integrado.

    Diferencias vs run_agent() del NB 04:
      1. Inyecta contexto histórico en el system prompt (memory.build_context_block)
      2. Usa dispatch_v2 (fix WinError 2)
      3. Guarda output de cada tool en tool_log
      4. Al finalizar: guarda sesión + alertas en memoria automáticamente
    """

    # ── 1. Cargar contexto histórico ──────────────────────────────────────
    context_block = memory.build_context_block()
    system_prompt = SYSTEM_PROMPT_BASE
    if context_block:
        system_prompt = SYSTEM_PROMPT_BASE + '\n\n' + context_block
        if verbose:
            print(f'📚 Historial cargado ({memory.count_sessions()} sesión/es previas)')
    else:
        if verbose:
            print('🆕 Primera sesión — memoria vacía')

    # ── 2. Inicializar loop ───────────────────────────────────────────────
    session_id = str(uuid.uuid4())[:8]
    messages   = [{'role': 'user', 'content': prompt}]
    tokens_in  = tokens_out = turns = 0
    final_text = ''
    tool_log   = []

    if verbose:
        print(f'USUARIO [{session_id}]: {prompt}')
        print('-' * 60)

    # ── 3. Loop agentico (idéntico al NB 04) ─────────────────────────────
    while turns < max_turns:
        turns += 1

        resp = client.messages.create(
            model      = MODEL,
            max_tokens = 4096,
            system     = system_prompt,
            tools      = TOOL_DEFS,
            messages   = messages,
        )

        tokens_in  += resp.usage.input_tokens
        tokens_out += resp.usage.output_tokens

        if resp.stop_reason == 'end_turn':
            final_text = ''.join(b.text for b in resp.content if b.type == 'text')
            if verbose:
                print(f'AGENTE: {final_text}')
            break

        tool_results = []
        for block in resp.content:
            if block.type == 'tool_use':
                if verbose:
                    print(f'  [T{turns}] {block.name} | {json.dumps(block.input)[:80]}')

                try:
                    output = dispatch_v2(block.name, block.input)
                except Exception as e:
                    output = f'ERROR: {e}'

                # ← Guardar output (nuevo vs NB 04 — necesario para extracción)
                tool_log.append({
                    'turn'  : turns,
                    'tool'  : block.name,
                    'input' : block.input,
                    'output': output,
                })

                if verbose:
                    print(f'  [T{turns}] Resultado: {str(output)[:100]}...')

                tool_results.append({
                    'type'       : 'tool_result',
                    'tool_use_id': block.id,
                    'content'    : str(output),
                })

        if not tool_results:
            final_text = ''.join(b.text for b in resp.content if b.type == 'text')
            break

        messages.append({'role': 'assistant', 'content': resp.content})
        messages.append({'role': 'user',      'content': tool_results})

    result = {
        'session_id': session_id,
        'final_text': final_text,
        'turns'     : turns,
        'tokens_in' : tokens_in,
        'tokens_out': tokens_out,
        'tool_log'  : tool_log,
    }

    # ── 4. Guardar en memoria ─────────────────────────────────────────────
    memory.save_session(session_id, prompt, result)

    states = extract_account_state(tool_log)
    if states:
        memory.save_alertas(session_id, states)
        if verbose:
            niveles = ', '.join(f"{s['cuenta_id']}:{s['nivel']}" for s in states)
            print(f'\n💾 Memoria guardada: [{niveles}] | Session: {session_id}')

    return result


print('run_agent_with_memory() listo.')

run_agent_with_memory() listo.


---
## DEMO — Sesión 1: memoria vacía

El agente no tiene historial. Analiza el estado actual y guarda todo en memoria.  
Es idéntico al NB 04 — pero ahora el resultado persiste.

In [8]:
# Reiniciar la memoria para el demo (borrar sesiones previas si existen)
# Comentar estas líneas si quieres acumular sesiones reales
import shutil
if MEMORY_DB.exists():
    MEMORY_DB.unlink()
    print('⚠️  Memoria anterior borrada para el demo.')

memory = MemoryStore(MEMORY_DB)
print(memory)

⚠️  Memoria anterior borrada para el demo.
<MemoryStore db=agente_tesoreria.db sessions=0>


In [9]:
# SESIÓN 1: análisis estándar — primera vez que el agente ve a este cliente
resultado_s1 = run_agent_with_memory(
    '¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo.',
    memory  = memory,
    verbose = True,
)

print()
print('=== MÉTRICAS SESIÓN 1 ===')
print(f'  Session ID: {resultado_s1["session_id"]}')
print(f'  Turns:      {resultado_s1["turns"]}')
print(f'  Tokens in:  {resultado_s1["tokens_in"]:,}')
print(f'  Tokens out: {resultado_s1["tokens_out"]:,}')
print(f'  Tools:      {[t["tool"] for t in resultado_s1["tool_log"]]}')

🆕 Primera sesión — memoria vacía
USUARIO [0838d860]: ¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo.
------------------------------------------------------------
  [T1] read_skill | {"skill_name": "reporte-semanal/SKILL.md"}
  [T1] Resultado: ---
name: reporte-semanal-tesoreria
description: >
  Estructura del reporte semanal de flujo de caja...
  [T1] read_skill | {"skill_name": "alerta-tesoreria/SKILL.md"}
  [T1] Resultado: ---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere a...
  [T2] bash_execute | {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect
  [T2] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [T3] bash_execute | {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect
  [T3] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [T4] bash_execute | {"command": ["cmd", "/c", "dir", "D

In [10]:
# Ver qué se guardó en memoria
print('=== MEMORIA DESPUÉS DE SESIÓN 1 ===\n')

print('Sesiones guardadas:')
for s in memory.get_recent_sessions():
    print(f'  [{s["fecha"][:16]}] {s["query"]}')
    print(f'  Turns: {s["turns"]} | Tokens: {s["tokens_in"]+s["tokens_out"]:,}')
    print(f'  Resumen: {s["summary"][:100]}...')
    print()

print('Estados de cuentas guardados:')
for a in memory.get_last_state_per_account():
    nivel_emoji = {'CRITICO': '🔴', 'ALTO': '🟠', 'MEDIO': '🟡', 'OK': '🟢'}
    emoji = nivel_emoji.get(a['nivel'], '⚪')
    print(f"  {a['cuenta_id']}: {emoji} {a['nivel']} | "
          f"Saldo: {a['saldo_actual']:,.0f} | "
          f"Flujo/día: {a['flujo_neto']:+.0f}")

=== MEMORIA DESPUÉS DE SESIÓN 1 ===

Sesiones guardadas:
  [2026-05-27T02:40] ¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo.
  Turns: 8 | Tokens: 30,182
  Resumen: ---  # 📊 Reporte de Tesorería — Semana del 26 de Mayo 2026  ## Resumen Ejecutivo  ✅ **Buenas noticia...

Estados de cuentas guardados:


---
## Enriquecer la memoria manualmente

El tesorero puede marcar acciones pendientes y configurar preferencias del cliente.  
Estas entradas se inyectan en el contexto de la próxima sesión.

In [11]:
# Después de revisar el output de Sesión 1, el tesorero registra:

# 1. Acción pendiente detectada en la sesión
memory.save_pending_action(
    session_id = resultado_s1['session_id'],
    accion     = 'Revisar calendario de pagos en USD para las próximas 2 semanas',
    cuenta_id  = 'CUENTA-003'
)

# 2. Configuración específica del cliente
memory.save_fact(
    key   = 'CUENTA-002.alerta_custom',
    value = 'Monitorear especialmente los días 13-15 de cada mes (planilla BBVA)'
)
memory.save_fact(
    key   = 'cliente.nombre',
    value = 'Constructora Andina S.A.C.'
)

print('✅ Memoria enriquecida con acción pendiente + 2 datos del cliente.')
print()
print('Acciones pendientes:')
for p in memory.get_pending_actions():
    print(f"  - [{p['cuenta_id']}] {p['accion']}")
print()
print('Client facts:')
for k, v in memory.get_client_facts().items():
    print(f'  {k}: {v}')

✅ Memoria enriquecida con acción pendiente + 2 datos del cliente.

Acciones pendientes:
  - [CUENTA-003] Revisar calendario de pagos en USD para las próximas 2 semanas

Client facts:
  CUENTA-002.alerta_custom: Monitorear especialmente los días 13-15 de cada mes (planilla BBVA)
  cliente.nombre: Constructora Andina S.A.C.


---
## DEMO — Sesión 2: el agente ya sabe quién eres

Esta es la diferencia entre el goldfish (NB 04) y el CFO interno (NB 05).

El agente arranca con el contexto de la Sesión 1 inyectado en el system prompt.  
Puede comparar el estado actual con el que conocía, y sabe qué acciones siguen pendientes.

In [12]:
# Ver el bloque de contexto que el agente va a recibir ANTES de ejecutar
print('=== CONTEXTO QUE EL AGENTE RECIBIRÁ EN SESIÓN 2 ===')
print()
ctx = memory.build_context_block()
print(ctx if ctx else '(vacío)')

=== CONTEXTO QUE EL AGENTE RECIBIRÁ EN SESIÓN 2 ===


## HISTORIAL DEL CLIENTE (referencia interna — no citar textualmente al usuario)

### Sesiones recientes:
- [2026-05-27] "¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo." → ---  # 📊 Reporte de Tesorería — Semana del 26 de Mayo 2026  ## Resumen Ejecutivo  ✅ **Buenas noticias:** Las **3 cuentas están en zona OK** — ninguna 

### Acciones pendientes (recomendadas, aún no completadas):
  - Revisar calendario de pagos en USD para las próximas 2 semanas [CUENTA-003] (desde: 2026-05-27)

### Configuración del cliente:
  - CUENTA-002.alerta_custom: Monitorear especialmente los días 13-15 de cada mes (planilla BBVA)
  - cliente.nombre: Constructora Andina S.A.C.



In [13]:
# SESIÓN 2: el cliente pregunta por cambios respecto a la sesión anterior
resultado_s2 = run_agent_with_memory(
    '¿Algo cambió respecto a la última revisión? '
    '¿La situación de CUENTA-003 mejoró o sigue igual?',
    memory  = memory,
    verbose = True,
)

print()
print('=== MÉTRICAS SESIÓN 2 ===')
print(f'  Session ID: {resultado_s2["session_id"]}')
print(f'  Turns:      {resultado_s2["turns"]}')
print(f'  Tokens in:  {resultado_s2["tokens_in"]:,}')
print(f'  Tokens out: {resultado_s2["tokens_out"]:,}')
print(f'  Tools:      {[t["tool"] for t in resultado_s2["tool_log"]]}')

📚 Historial cargado (1 sesión/es previas)
USUARIO [31248bab]: ¿Algo cambió respecto a la última revisión? ¿La situación de CUENTA-003 mejoró o sigue igual?
------------------------------------------------------------
  [T1] bash_execute | {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect
  [T1] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [T1] read_skill | {"skill_name": "alerta-tesoreria/SKILL.md"}
  [T1] Resultado: ---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere a...
  [T2] bash_execute | {"command": ["d:/Proyecto_Gabriel/.venv/Scripts/python.exe", "D:/Proyecto_Gabrie
  [T2] Resultado: [
  {
    "cuenta_id": "CUENTA-001",
    "moneda": "PEN",
    "saldo_actual": 356608.41,
    "saldo_...
AGENTE: Tengo los datos actualizados. Aquí va la comparativa:

---

## 📊 Comparativa vs. Revisión del 27-May — Constructora Andina S.A.C.

### CUENTA-003 (USD) — ¿Mejoró o si

---
## DEMO — Sesión 3: historial acumulado

Con 2 sesiones en memoria, el agente puede detectar patrones temporales.  
Esta es la pregunta del CFO que un goldfish no puede responder.

In [14]:
# Ver el contexto acumulado de 2 sesiones
print(f'Sesiones en memoria: {memory.count_sessions()}')
print()
print('=== CONTEXTO PARA SESIÓN 3 ===')
print(memory.build_context_block())

Sesiones en memoria: 2

=== CONTEXTO PARA SESIÓN 3 ===

## HISTORIAL DEL CLIENTE (referencia interna — no citar textualmente al usuario)

### Sesiones recientes:
- [2026-05-27] "¿Algo cambió respecto a la última revisión? ¿La situación de CUENTA-003 mejoró o sigue igual?" → Tengo los datos actualizados. Aquí va la comparativa:  ---  ## 📊 Comparativa vs. Revisión del 27-May — Constructora Andina S.A.C.  ### CUENTA-003 (USD
- [2026-05-27] "¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo." → ---  # 📊 Reporte de Tesorería — Semana del 26 de Mayo 2026  ## Resumen Ejecutivo  ✅ **Buenas noticias:** Las **3 cuentas están en zona OK** — ninguna 

### Último estado conocido por cuenta:
  CUENTA-001: 🟢 OK | Saldo: 356,608 | Flujo/día: +1969 | Registrado: 2026-05-27
  CUENTA-002: 🟢 OK | Saldo: 224,331 | Flujo/día: +1289 | Registrado: 2026-05-27
  CUENTA-003: 🟢 OK | Saldo: 118,639 | Flujo/día: +187 | Registrado: 2026-05-27

### Acciones pendientes (recomendadas, aún no completadas):
  

In [15]:
# SESIÓN 3: pregunta de síntesis histórica
resultado_s3 = run_agent_with_memory(
    'Dame un resumen de las revisiones de esta semana. '
    '¿Qué tendencias ves? ¿Hay alguna acción pendiente que aún no hayamos completado?',
    memory  = memory,
    verbose = True,
)

print()
print('=== MÉTRICAS SESIÓN 3 ===')
print(f'  Session ID: {resultado_s3["session_id"]}')
print(f'  Turns:      {resultado_s3["turns"]}')
print(f'  Tokens in:  {resultado_s3["tokens_in"]:,}')
print(f'  Tokens out: {resultado_s3["tokens_out"]:,}')
print(f'  Tools:      {[t["tool"] for t in resultado_s3["tool_log"]]}')

📚 Historial cargado (2 sesión/es previas)
USUARIO [733e253f]: Dame un resumen de las revisiones de esta semana. ¿Qué tendencias ves? ¿Hay alguna acción pendiente que aún no hayamos completado?
------------------------------------------------------------
  [T1] bash_execute | {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect
  [T1] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [T1] read_skill | {"skill_name": "alerta-tesoreria/SKILL.md"}
  [T1] Resultado: ---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere a...
AGENTE: Hubo un error al ejecutar el script de datos en vivo. No te preocupes — tengo el historial registrado de las sesiones de esta semana y puedo darte un análisis completo basado en eso.

---

# 📋 Resumen Semanal — Constructora Andina S.A.C.
**Semana del 26 al 27 de Mayo, 2026**

---

## 🔁 Revisiones realizadas esta semana

Se hicieron **2 revisiones docum

---
## Inspección completa de la memoria

Ver el estado final de todas las tablas SQLite después de las 3 sesiones.

In [16]:
import pandas as pd

def dump_memory(memory: MemoryStore) -> None:
    """Muestra el estado completo de todas las tablas."""
    with sqlite3.connect(memory.db_path) as conn:
        tables = ['sessions', 'alertas', 'client_facts', 'pending_actions']
        for t in tables:
            df = pd.read_sql_query(f'SELECT * FROM {t}', conn)
            print(f'\n=== {t.upper()} ({len(df)} filas) ===')
            if df.empty:
                print('  (vacía)')
            else:
                # Truncar columnas largas para mejor visualización
                for col in ['summary', 'query', 'accion']:
                    if col in df.columns:
                        df[col] = df[col].astype(str).str[:60] + '...'
                print(df.to_string(index=False))


dump_memory(memory)


=== SESSIONS (3 filas) ===
      id                      fecha                                                           query                                                         summary  turns  tokens_in  tokens_out
0838d860 2026-05-27T02:40:48.246619 ¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo... ---  # 📊 Reporte de Tesorería — Semana del 26 de Mayo 2026  ...      8      28638        1544
31248bab 2026-05-27T02:41:10.176853 ¿Algo cambió respecto a la última revisión? ¿La situación de... Tengo los datos actualizados. Aquí va la comparativa:  ---  ...      3       7339        1058
733e253f 2026-05-27T02:41:34.741274 Dame un resumen de las revisiones de esta semana. ¿Qué tende... Hubo un error al ejecutar el script de datos en vivo. No te ...      2       4712        1261

=== ALERTAS (3 filas) ===
 id session_id  cuenta_id nivel  saldo_actual  dias_de_caja  flujo_neto                      fecha
  1   31248bab CUENTA-001    OK     356608.41         999.0     1968.9

---
## Métricas comparativas

El costo de la memoria no es cero — el contexto histórico agrega tokens al prompt.  
Pero el overhead es controlado y el valor diferencial es claro.

In [17]:
resultados = [resultado_s1, resultado_s2, resultado_s3]
labels     = ['Sesión 1 (sin memoria)', 'Sesión 2 (1 sesión previa)', 'Sesión 3 (2 sesiones previas)']

df_metricas = pd.DataFrame([
    {
        'Sesión'    : labels[i],
        'Turns'     : r['turns'],
        'Tokens in' : r['tokens_in'],
        'Tokens out': r['tokens_out'],
        'Total'     : r['tokens_in'] + r['tokens_out'],
        'Tool calls': len(r['tool_log']),
    }
    for i, r in enumerate(resultados)
])

print('=== MÉTRICAS DE LAS 3 SESIONES ===\n')
print(df_metricas.to_string(index=False))

# Costo
PRICE_IN  = 3.00  / 1_000_000   # USD por token input
PRICE_OUT = 15.00 / 1_000_000   # USD por token output

print()
costo_total = 0
for i, r in enumerate(resultados):
    costo = r['tokens_in'] * PRICE_IN + r['tokens_out'] * PRICE_OUT
    costo_total += costo
    ctx_size = len(memory.build_context_block())
    print(f'{labels[i]}: USD ${costo:.4f}')

print(f'\nTotal 3 sesiones: USD ${costo_total:.4f}')
print(f'Costo por sesión: USD ${costo_total/3:.4f}')

=== MÉTRICAS DE LAS 3 SESIONES ===

                       Sesión  Turns  Tokens in  Tokens out  Total  Tool calls
       Sesión 1 (sin memoria)      8      28638        1544  30182           9
   Sesión 2 (1 sesión previa)      3       7339        1058   8397           3
Sesión 3 (2 sesiones previas)      2       4712        1261   5973           2

Sesión 1 (sin memoria): USD $0.1091
Sesión 2 (1 sesión previa): USD $0.0379
Sesión 3 (2 sesiones previas): USD $0.0331

Total 3 sesiones: USD $0.1800
Costo por sesión: USD $0.0600


---
## Resumen — Arquitectura completa del agente con memoria

```
NB 01 → Datos simulados (CSV base, 270 filas)
   ↓
NB 02 → Forecast de caja (batch_dias_de_caja.py, rolling_mean.py)
   ↓
NB 03 → Los 3 SKILLs .md (políticas: forecast / alertas / reporte)
   ↓
NB 04 → Loop agentico base (Messages API, 2 tools)
   ↓
NB 05 → Memory Store (SQLite, CFO interno)
```

| Componente | Responsabilidad | Archivo |
|---|---|---|
| MemoryStore | Persistencia SQLite entre sesiones | NB 05 |
| dispatch_v2 | Ejecutar tools (fix WinError 2) | NB 05 |
| extract_account_state | Parsear outputs JSON del tool_log | NB 05 |
| run_agent_with_memory | Loop + memoria integrada | NB 05 |
| build_context_block | Inyección de historial en prompt | MemoryStore |

### El diferenciador comercial

| Agente sin memoria (NB 04) | CFO interno (NB 05) |
|---|---|
| Cada sesión empieza desde cero | Recuerda KPIs de sesiones previas |
| No puede comparar semana vs semana | Detecta tendencias en el tiempo |
| No registra acciones pendientes | Trackea compromisos no completados |
| No conoce preferencias del cliente | Almacena configuración por cuenta |
| Análisis puntual | Narrativa continua |

### Estado actual del proyecto

| Capa | Notebook | Estado |
|---|---|---|
| 1 — Datos | NB 01 | ✅ |
| 2 — Forecast | NB 02 | ✅ |
| 3 — SKILLs | NB 03 | ✅ |
| 4 — Loop base | NB 04 | ✅ |
| 5 — Memoria | NB 05 | ✅ |

### Próximo paso posible

**Capa 6 — Interfaz de cliente:**  
Un script `tesorero.py` o Streamlit app que envuelva `run_agent_with_memory()`  
y exponga la interfaz del CFO interno al cliente final sin que vea el código.

---

*NB 05 completado. La memoria persiste en `Skill_financiero/data/memory/agente_tesoreria.db`.*